# Clase 136 — Forecasting de series con RNN

Aplicamos RNN/LSTM/GRU a **forecasting de series temporales** con un flujo correcto:
**split temporal** (nunca aleatorio), ventanas con `timeseries_dataset_from_array`,
comparación contra un **baseline naïve** y métricas estándar (MAE).

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`. Se ejecuta en Colab con GPU.

## 1. Serie sintética y split TEMPORAL (sin shuffle)

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
r = np.random.default_rng(42)
t = np.arange(2000)
serie = (np.sin(2 * np.pi * t / 50) + 0.3 * np.sin(2 * np.pi * t / 13)
         + r.normal(0, 0.1, t.size)).astype("float32")

n = len(serie)
tr_end, val_end = int(n * 0.7), int(n * 0.85)
train, val, test = serie[:tr_end], serie[tr_end:val_end], serie[val_end:]  # 70/15/15 temporal
mu, sigma = train.mean(), train.std()          # stats SOLO de train (sin leakage)
train_n = (train - mu) / sigma
val_n = (val - mu) / sigma
test_n = (test - mu) / sigma
print("tamaños:", train.shape, val.shape, test.shape)

## 2. Ventanas con `timeseries_dataset_from_array`

In [ ]:
T = 30   # lookback
def hacer_ds(arr, batch=32):
    ds = keras.utils.timeseries_dataset_from_array(
        data=arr[:-1],           # inputs
        targets=arr[T:],         # target = valor justo después de la ventana
        sequence_length=T,
        batch_size=batch,
    )
    return ds.map(lambda x, y: (x[..., None], y))   # (batch, T, 1)

ds_tr = hacer_ds(train_n)
ds_val = hacer_ds(val_n)
for xb, yb in ds_tr.take(1):
    print("batch X:", xb.shape, "| batch y:", yb.shape)

## 3. Baseline naïve: `ŷ_t = y_{t-1}`

In [ ]:
def mae(a, b):
    return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))

naive_pred = test_n[T - 1:-1]   # último valor de cada ventana
naive_true = test_n[T:]
print("MAE naïve (test):", round(mae(naive_true, naive_pred), 4))
# Siempre hay que superar este baseline para justificar la RNN.

## 4. RNN de forecasting

In [ ]:
modelo = keras.Sequential([
    keras.Input(shape=(T, 1)),
    layers.SimpleRNN(32),
    layers.Dense(1),
])
modelo.compile(optimizer="adam", loss="mae")
modelo.fit(ds_tr, validation_data=ds_val, epochs=5, verbose=2)

## 5. Deep RNN: apilar LSTM

In [ ]:
profundo = keras.Sequential([
    keras.Input(shape=(T, 1)),
    layers.LSTM(32, return_sequences=True),   # devuelve la secuencia para la siguiente capa
    layers.LSTM(32),
    layers.Dense(1),
])
profundo.compile(optimizer="adam", loss="mae")
profundo.summary()

## 6. Multi-step directo: predecir un horizonte completo

In [ ]:
H = 7   # horizonte
multi = keras.Sequential([
    keras.Input(shape=(T, 1)),
    layers.LSTM(32),
    layers.Dense(H),     # H salidas simultáneas (directo, no recursivo)
])
multi.compile(optimizer="adam", loss="mae")
print("salida multi-step:", multi.output_shape)   # (None, 7)

## Ejercicios

1. **Split temporal**: separá 70/15/15 sin mezclar y verificá que val/test son el
   período más reciente.
2. **Baseline naïve**: reportá su MAE y usalo como piso a superar.
3. **LSTM vs GRU**: entrená ambos con el mismo `ds_tr` y compará MAE y velocidad.
4. **Multi-step**: compará la predicción directa (`Dense(7)`) contra la recursiva
   (realimentar 1 paso 7 veces).

## Conclusiones

- El **split temporal** es obligatorio: mezclar filtra el futuro al pasado (leakage).
- Normalizá con estadísticas **solo de train**.
- `timeseries_dataset_from_array` arma las ventanas `(batch, T, features)` automáticamente.
- Toda RNN debe compararse contra el **baseline naïve**; si no lo supera, no aporta.
- Multi-step **directo** (`Dense(H)`) evita la acumulación de error del recursivo.
- Para series chicas ARIMA/ETS pueden empatar; DL gana con series largas y features.